# 10 - Refutação e robustez

## Setup

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings("ignore")

df = pd.read_parquet("../data/processed/analysis_dataset.parquet")

# Winsorização aplicada nas estimações anteriores (mesmos limites do notebook 08)
lo, hi = df["delta_y"].quantile([0.01, 0.99])
df["dy_w"] = df["delta_y"].clip(lo, hi)

print(f"Base analítica: {len(df)} pares | {df['within_paper_group_id'].nunique()} grupos")
print(f"Estratégias: {df['strategy'].value_counts().to_dict()}")
print(f"Winsorização: [{lo:.3f}, {hi:.3f}]")

Base analítica: 425 pares | 279 grupos
Estratégias: {'other': 158, 'oversampling': 156, 'cost_sensitive': 79, 'undersampling': 32}
Winsorização: [-0.619, 11.134]


---
## Parte 1 — Testes de independência condicional (CI tests)

O DAG assume que todas as variáveis de **Z = {IR, NC, S, T, M}** são nós raiz sem arestas diretas entre si,
implicando 10 independências marginais. Testamos cada par com qui-quadrado de Pearson
(variáveis já categorizadas) e correção de Bonferroni (α = 0,05 / 10 = 0,005).

Referência: Lecture 6, Part II — Refuting a Causal DAG.

In [2]:
ALPHA = 0.05
N_TESTS = 10
BONFERRONI_ALPHA = ALPHA / N_TESTS

pairs = [
    ("ir_bin",       "nclasses_bin",  "IR ⊥ NC"),
    ("ir_bin",       "size_bin",      "IR ⊥ S"),
    ("ir_bin",       "task3",         "IR ⊥ T"),
    ("ir_bin",       "model_family3", "IR ⊥ M"),
    ("nclasses_bin", "size_bin",      "NC ⊥ S"),
    ("nclasses_bin", "task3",         "NC ⊥ T"),
    ("nclasses_bin", "model_family3", "NC ⊥ M"),
    ("size_bin",     "task3",         "S ⊥ T"),
    ("size_bin",     "model_family3", "S ⊥ M"),
    ("task3",        "model_family3", "T ⊥ M"),
]

print(f"Threshold Bonferroni: α = {BONFERRONI_ALPHA:.4f}\n")
print(f"{'Asserção':<12}  {'χ²':>8}  {'gl':>4}  {'p-value':>10}  Veredicto")
print("-" * 58)

results = []
for col1, col2, label in pairs:
    ct = pd.crosstab(df[col1], df[col2])
    chi2, p, dof, _ = chi2_contingency(ct)
    verdict = "FALHA ✗" if p < BONFERRONI_ALPHA else "sobrevive ✓"
    print(f"{label:<12}  {chi2:>8.1f}  {dof:>4}  {p:>10.4f}  {verdict}")
    results.append({"assercao": label, "chi2": chi2, "gl": dof, "p_value": p,
                    "veredicto": "refutada" if p < BONFERRONI_ALPHA else "sobrevive"})

results_df = pd.DataFrame(results)
n_failed = (results_df["veredicto"] == "refutada").sum()
print(f"\n→ {n_failed}/{N_TESTS} asserções refutadas")

Threshold Bonferroni: α = 0.0050

Asserção            χ²    gl     p-value  Veredicto
----------------------------------------------------------
IR ⊥ NC          107.4    12      0.0000  FALHA ✗
IR ⊥ S            93.3    12      0.0000  FALHA ✗
IR ⊥ T            85.6     8      0.0000  FALHA ✗
IR ⊥ M           100.7    12      0.0000  FALHA ✗
NC ⊥ S           194.8     9      0.0000  FALHA ✗
NC ⊥ T           252.0     6      0.0000  FALHA ✗
NC ⊥ M           174.3     9      0.0000  FALHA ✗
S ⊥ T            152.1     6      0.0000  FALHA ✗
S ⊥ M            190.8     9      0.0000  FALHA ✗
T ⊥ M            149.7     6      0.0000  FALHA ✗

→ 10/10 asserções refutadas


---
## Parte 2 — Refutadores do efeito estimado

Aplicamos três refutadores ao efeito do **oversampling** (estratégia principal, n=156) usando o estimador g-formula
(regressão linear + padronização sobre a distribuição marginal de Z), reproduzindo a lógica do notebook 08.

- **Placebo treatment**: permuta aleatória dos rótulos de estratégia — efeito deveria cair a zero (mas não cairá aqui por viés de publicação).
- **Random common cause**: adiciona um confundidor sintético gaussiano — efeito não deve se alterar.
- **Data subset**: reestima em subamostras de 80% dos grupos — verifica estabilidade.

In [3]:
COVARS = ["ir_bin", "nclasses_bin", "size_bin", "task3", "model_family3"]
STRATEGY = "oversampling"
N_BOOT = 500
np.random.seed(42)


def gformula_ate(df_train, df_marginal, covars, strategy):
    """G-formula: fit on strategy rows, standardize over marginal Z distribution."""
    sub = df_train[df_train["strategy"] == strategy].copy()
    X_train = pd.get_dummies(sub[covars], drop_first=False)
    X_marg = pd.get_dummies(df_marginal[covars], drop_first=False).reindex(
        columns=X_train.columns, fill_value=0
    )
    reg = LinearRegression().fit(X_train, sub["dy_w"])
    return reg.predict(X_marg).mean()


# ── Observed ATE ─────────────────────────────────────────────────────────────
obs_ate = gformula_ate(df, df, COVARS, STRATEGY)
print(f"ATE observado ({STRATEGY}): {obs_ate:.4f}  ({obs_ate*100:.1f}%)\n")

# ── Cluster bootstrap (IC do ATE) ─────────────────────────────────────────────
groups = df["within_paper_group_id"].unique()
boot_ates = []
for _ in range(N_BOOT):
    sg = np.random.choice(groups, size=len(groups), replace=True)
    bdf = pd.concat([df[df["within_paper_group_id"] == g] for g in sg])
    try:
        boot_ates.append(gformula_ate(bdf, bdf, COVARS, STRATEGY))
    except Exception:
        pass
ic_lo, ic_hi = np.percentile(boot_ates, [2.5, 97.5])
print(f"Bootstrap IC 95%: [{ic_lo:.4f}; {ic_hi:.4f}]  "
      f"([{ic_lo*100:.1f}%; {ic_hi*100:.1f}%])\n")

ATE observado (oversampling): 0.4041  (40.4%)



Bootstrap IC 95%: [0.1182; 0.8047]  ([11.8%; 80.5%])



In [4]:
# ── Refutador 1: Placebo treatment ───────────────────────────────────────────
placebo_ates = []
for _ in range(N_BOOT):
    df_p = df.copy()
    df_p["strategy"] = np.random.permutation(df_p["strategy"].values)
    try:
        placebo_ates.append(gformula_ate(df_p, df_p, COVARS, STRATEGY))
    except Exception:
        pass
print(f"Placebo treatment ATE: {np.mean(placebo_ates):.4f} ± {np.std(placebo_ates):.4f}")
print(f"  Interpretação: valor não-nulo por viés de publicação "
      f"(todas as estratégias têm ΔY > 0 na base)\n")

# ── Refutador 2: Random common cause ─────────────────────────────────────────
sub = df[df["strategy"] == STRATEGY].copy()
rcc_ates = []
for _ in range(N_BOOT):
    sub_r = sub.copy()
    sub_r["rc"] = np.random.normal(size=len(sub_r))
    X_rc = pd.get_dummies(sub_r[COVARS], drop_first=False)
    X_rc["rc"] = sub_r["rc"]
    df_mg = df.copy()
    df_mg["rc"] = np.random.normal(size=len(df_mg))
    X_mg = pd.get_dummies(df_mg[COVARS], drop_first=False).reindex(
        columns=pd.get_dummies(sub[COVARS], drop_first=False).columns, fill_value=0
    )
    X_mg["rc"] = df_mg["rc"]
    try:
        reg_r = LinearRegression().fit(X_rc, sub_r["dy_w"])
        rcc_ates.append(reg_r.predict(X_mg).mean())
    except Exception:
        pass
rcc_mean = np.mean(rcc_ates)
print(f"Random common cause ATE: {rcc_mean:.4f}  "
      f"(mudança: {abs(rcc_mean - obs_ate)*100:.2f}pp)\n")

# ── Refutador 3: Data subset ──────────────────────────────────────────────────
subset_ates = []
for _ in range(N_BOOT):
    sg = np.random.choice(groups, size=int(0.8 * len(groups)), replace=False)
    sdf = df[df["within_paper_group_id"].isin(sg)].copy()
    try:
        subset_ates.append(gformula_ate(sdf, sdf, COVARS, STRATEGY))
    except Exception:
        pass
sub_lo, sub_hi = np.percentile(subset_ates, [2.5, 97.5])
print(f"Data subset ATE (80% grupos): [{sub_lo:.4f}; {sub_hi:.4f}]  "
      f"([{sub_lo*100:.1f}%; {sub_hi*100:.1f}%])")

Placebo treatment ATE: 0.3299 ± 0.0923
  Interpretação: valor não-nulo por viés de publicação (todas as estratégias têm ΔY > 0 na base)



Random common cause ATE: 0.4029  (mudança: 0.12pp)



Data subset ATE (80% grupos): [0.2316; 0.5191]  ([23.2%; 51.9%])


---
## Resumo

| Teste | Resultado | Interpretação |
|---|---|---|
| **CI tests (10 pares de Z)** | 10/10 refutadas | Z variables são correlacionadas — DAG simplificado, mas backdoor ainda válido |
| **Placebo treatment** | ~34% (não cai a 0) | Viés de publicação: ΔY > 0 para toda a base |
| **Random common cause** | mudança < 0,15pp | Estimador insensível a variáveis irrelevantes |
| **Data subset (80%)** | IC contém ATE original | Resultado estável a variações na amostra |